# 모델 양자화 (Quantization) - 실습 코드 2: 직접 구현: Min-Max / Affine 양자화

- Tutorial ID: `expand-quantization`
- Tutorial: 모델 양자화 (Quantization)
- Section ID: `expand-quantization-code-2`
- Section: 실습 코드 2: 직접 구현: Min-Max / Affine 양자화


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: 직접 구현: Min-Max / Affine 양자화
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# ── 1. Min-Max 양자화 (대칭) ──
def quantize_symmetric(tensor, n_bits=4):
    """대칭 양자화: 0을 중심으로 [-2^(b-1)+1, 2^(b-1)-1] 범위로 매핑"""
    max_val = torch.max(torch.abs(tensor))
    scale = max_val / (2 ** (n_bits - 1) - 1)
    quantized = torch.round(tensor / scale).clamp(
        -(2 ** (n_bits - 1)), 2 ** (n_bits - 1) - 1
    )
    dequantized = quantized * scale
    return quantized.to(torch.int8), dequantized, scale

# ── 2. Affine 양자화 (비대칭) ──
def quantize_affine(tensor, n_bits=4):
    """비대칭 양자화: [min, max]를 [0, 2^b-1]로 매핑"""
    min_val, max_val = tensor.min(), tensor.max()
    scale = (max_val - min_val) / (2 ** n_bits - 1)
    zero_point = torch.round(-min_val / scale)
    
    quantized = torch.round(tensor / scale + zero_point).clamp(0, 2 ** n_bits - 1)
    dequantized = (quantized - zero_point) * scale
    return quantized.to(torch.uint8), dequantized, scale, zero_point

# ── 3. 중요 채널 보존 양자화 (AWQ 스타일) ──
def quantize_awq_style(weight, activation, n_bits=4, alpha=0.5):
    """AWQ 스타일: activation 크기가 큰 채널은 보존"""
    # 채널별 activation 크기 계산
    channel_importance = activation.abs().mean(dim=0)  # (out_features,)
    threshold = torch.quantile(channel_importance, alpha)
    
    # 중요 채널 마스크
        important_mask = channel_importance > threshold
    
    # 중요 채널은 더 높은 정밀도로 양자화
    weight_quantized = weight.clone()
        for i in range(weight.shape[0]):
        if important_mask[i]:
            # 중요: 8-bit 양자화
            _, deq, _ = quantize_symmetric(weight[i:i+1], n_bits=8)
        else:
            # 덜 중요: 4-bit 양자화
            _, deq, _ = quantize_symmetric(weight[i:i+1], n_bits=n_bits)
        weight_quantized[i] = deq.squeeze()
    
    return weight_quantized, important_mask

# ── 실험: 양자화 오차 비교 ──
torch.manual_seed(42)
weight = torch.randn(256, 512) * 0.5  # 가중치 행렬
activation = torch.randn(1, 512)       # 활성화

print("=== 양자화 방법별 오차 비교 ===\n")

# Min-Max 8-bit
_, deq_8, _ = quantize_symmetric(weight, n_bits=8)
error_8 = (weight - deq_8).abs().mean().item()
mem_8 = weight.numel() * 1 / 1e6  # 1 byte per param
print(f"INT8 대칭 양자화: MSE={error_8:.6f}, 메모리={mem_8:.2f} MB")

# Min-Max 4-bit
_, deq_4, _ = quantize_symmetric(weight, n_bits=4)
error_4 = (weight - deq_4).abs().mean().item()
mem_4 = weight.numel() * 0.5 / 1e6  # 0.5 byte per param
print(f"INT4 대칭 양자화: MSE={error_4:.6f}, 메모리={mem_4:.2f} MB")

# Affine 4-bit
_, deq_aff, _, _ = quantize_affine(weight, n_bits=4)
error_aff = (weight - deq_aff).abs().mean().item()
print(f"INT4 비대칭 양자화: MSE={error_aff:.6f}, 메모리={mem_4:.2f} MB")

# AWQ 스타일
deq_awq, mask = quantize_awq_style(weight, activation, n_bits=4)
error_awq = (weight - deq_awq).abs().mean().item()
print(f"AWQ 스타일:        MSE={error_awq:.6f} (중요 채널 {mask.sum()}/{len(mask)}개 보존)")

print(f"\n원래 FP32: MSE=0.000000, 메모리={weight.numel() * 4 / 1e6:.2f} MB")
print(f"\n→ AWQ가 INT4보다 오차가 적음: 중요 채널을 높은 정밀도로 보존!")